
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# LAB - Batch Deployment

Welcome to the "Batch Deployment" lab! This lab focuses on batch deployment of machine learning models using Databricks. You will engage in tasks related to model inference, model registry, and explore performance results for feature such as Liquid Clustering using `CLUSTER BY`.

**Learning Objectives:**

By the end of this lab, you will be able to;

+ **Task 1: Load Dataset**
    + Load Dataset
    + Split the dataset into features and response sets

+ **Task 2: Inference with feature table**

    + Create Feature Table
    + Setup Feature Lookups
    + Fit and Register a Model with UC using Feature Table
    + Perform batch inference using Feature Engineering's  **`score_batch`** method.

+ **Task 3: Assess Liquid Clustering:**

    + Evaluate the performance results for specific optimization techniques:
        + Liquid Clustering


## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong> — How to select an environment version: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/azure/databricks/compute/serverless/dependencies#select-an-environment-version" style="color: #1976D2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">GCP</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>


## Classroom Setup

Before starting the Lab, run the provided classroom setup script. This script will define configuration variables necessary for the lab. Execute the following cell:

**Other Conventions:**

Throughout this Lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
%pip install -U databricks-sdk databricks-feature-engineering mlflow==3.12.0
%pip install mlflow-skinny[databricks]==3.12.0

dbutils.library.restartPython()

In [0]:
print(f"Username:Sandutta")
print(f"Catalog Name: dbx_catalog")
print(f"Schema Name:   dbx_schema")
print(f"Working Directory:/Volumes/databricks_telco_customer_dataset/v01/telco")
print(f"User DB Location:dbx_catalog.dbx_schema")

## Task 1: Load Dataset

+ Load a dataset:
  + Define the dataset path
  + Define the primary key (`customerID`), response variable (`Churn`), and feature variables (`SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`) for further processing.
  + Read the dataset, casting relevant columns to the correct data types, and drop any rows with missing values
+ Split the dataset into training and testing sets
  + Separate the features and the response for the training set


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
## Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn-missing' # CSV file name
dataset_p_telco = f"/Volumes/databricks_telco_customer_dataset/v01/telco/{csv_name}.csv" # Full path

## Features to use
primary_key = "customerID"
response = "Churn"
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

## Read dataset (and drop nan)
telco_df = spark.read.csv(dataset_p_telco, inferSchema=True, header=True, multiLine=True, escape='"')\
            .withColumn("TotalCharges", F.expr("try_cast(trim(TotalCharges) as double)"))\
            .withColumn("SeniorCitizen", col("SeniorCitizen").cast('double'))\
            .withColumn("Tenure", col("tenure").cast('double'))\
            .na.drop(how='any')

## Split with 80 percent of the data in train_df and 20 percent of the data in test_df
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

## Separate features and ground-truth
features_df = train_df.select(primary_key, *features)
response_df = train_df.select(primary_key, response)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1: Load Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql.functions import col

## Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn-missing' # CSV file name
dataset_p_telco = f"{DA.paths.datasets.telco}/{shared_volume_name}/{csv_name}.csv" # Full path

## features to use
primary_key = "customerID"
response = "Churn"
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

## Read dataset (and drop nan)
telco_df = spark.read.csv(dataset_p_telco, inferSchema=True, header=True, multiLine=True, escape='"')\
            .withColumn("TotalCharges", F.expr("try_cast(trim(TotalCharges) as double)"))\
            .withColumn("SeniorCitizen", col("SeniorCitizen").cast('double'))\
            .withColumn("Tenure", col("tenure").cast('double'))\
            .na.drop(how='any')

## Split with 80 percent of the data in train_df and 20 percent of the data in test_df
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

## Separate features and ground-truth
features_df = train_df.select(primary_key, *features)
response_df = train_df.select(primary_key, response)
```

</div>
</details>

##Task 2: Inference with feature table
In this task, you will perform batch inference using a feature table. Follow the steps below:

+ **Step 1: Create Feature Table**

+ **Step 2: Setup Feature Lookups**

+ **Step 3: Fit and Register a Model with UC using Feature Table**

+ **Step 4: Use the Model for Inference**


**Step 1: Create Feature Table**
  + Begin by creating a feature table that incorporates the relevant features for inference. This involves selecting the appropriate columns, performing any necessary transformations, and storing the resulting data in a feature table.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

## Prepare feature set
features_df_all = telco_df.select(primary_key, *features)

## Feature table definition
fe = FeatureEngineeringClient()
feature_table_name = "dbx_catalog.dbx_schema.features"

## Drop table if exists
try:
    fe.drop_table(name=feature_table_name)
except:
    pass

## Create feature table
fe.create_table(
    name=feature_table_name,
    df=features_df_all,
    primary_keys=[primary_key],
    description="Lab feature table"
)



<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Step 1: Create Feature Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.feature_engineering import FeatureEngineeringClient

## Prepare feature set
features_df_all = telco_df.select(primary_key, *features)

## Feature table definition
fe = FeatureEngineeringClient()
feature_table_name = f"{DA.catalog_name}.{DA.schema_name}.features"

## Drop table if exists
try:
    fe.drop_table(name=feature_table_name)
except:
    pass

## Create feature table
fe.create_table(
    name=feature_table_name,
    df=features_df_all,
    primary_keys=[primary_key],
    description="Lab feature table"
)
```

</div>
</details>

**Step 2: Setup Feature Lookups**
  + Set up a feature lookup to create a training set from the feature table. 
  + Specify the `lookup_key` based on the columns that uniquely identify records in your feature table.

In [0]:
from databricks.feature_engineering import FeatureLookup

fl_handle =FeatureLookup(
    table_name=feature_table_name,
    lookup_key=[primary_key]
)

## Create a training set based on feature lookup
training_set_spec = fe.create_training_set(
    df=response_df,
    label=response,
    feature_lookups=[fl_handle],
    exclude_columns=[primary_key]
)

## Load training dataframe based on defined feature-lookup specification
training_df = training_set_spec.load_df()


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Step 2: Setup Feature Lookups
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from databricks.feature_engineering import FeatureLookup

fl_handle = FeatureLookup(
    table_name=feature_table_name,
    lookup_key=[primary_key]
)

##  Create a training set based on feature lookup
training_set_spec = fe.create_training_set(
    df=response_df,
    label=response,
    feature_lookups=[fl_handle],
    exclude_columns=[primary_key]
)

## Load training dataframe based on defined feature-lookup specification
training_df = training_set_spec.load_df()
```

</div>
</details>

**Step 3: Fit and Register a Model with UC using Feature Table**
  + Fit and register a Machine Learning Model using the created training set.
  + Train a model on the training set and register it in the model registry.

In [0]:
import mlflow
import warnings
from mlflow.types.utils import _infer_schema

## Point to UC model registry
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

## Helper function that we will use for getting latest version of a model
def get_latest_model_version(model_name):
    """Helper function to get latest model version"""
    model_version_infos = client.search_model_versions("name = '%s'" % model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

## Train a sklearn Decision Tree Classification model
from sklearn.tree import DecisionTreeClassifier

## Convert data to pandas dataframes
X_train_pdf = training_df.drop(primary_key, response).toPandas()
Y_train_pdf = training_df.select(response).toPandas()
clf = DecisionTreeClassifier(max_depth=3, random_state=42)

## End the active MLflow run before starting a new one
mlflow.end_run()

with mlflow.start_run(run_name="Model-Batch-Deployment-lab-With-FS") as mlflow_run:

    ## Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True)
    
    clf.fit(X_train_pdf,Y_train_pdf)

    ## Infer output schema
    try:
        output_schema = _infer_schema(Y_train_pdf)
    except Exception as e:
        warnings.warn(f"Could not infer model output schema: {e}")
        output_schema = None

    model_name = "dbx_catalog.dbx_schema.ml_model_dc"
    
    ## Log using feature engineering client and push to registry
    fe.log_model(
        model=clf,
        artifact_path="decision_tree",
        flavor=mlflow.sklearn,
        training_set=training_set_spec,
        output_schema=output_schema,
        registered_model_name= model_name
    )

    ## Set model alias (i.e. Champion)
    client.set_registered_model_alias(model_name, "Champion", get_latest_model_version(model_name))


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Step 3: Fit and Register a Model with UC using Feature Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import mlflow
import warnings
from mlflow.types.utils import _infer_schema

## Point to UC model registry
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

## helper function that we will use for getting latest version of a model
def get_latest_model_version(model_name):
    """Helper function to get latest model version"""
    model_version_infos = client.search_model_versions("name = '%s'" % model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

## Train a sklearn Decision Tree Classification model
from sklearn.tree import DecisionTreeClassifier

## Convert data to pandas dataframes
X_train_pdf = training_df.drop(primary_key, response).toPandas()
Y_train_pdf = training_df.select(response).toPandas()
clf = DecisionTreeClassifier(max_depth=3, random_state=42)

## End the active MLflow run before starting a new one
mlflow.end_run()

with mlflow.start_run(run_name="Model-Batch-Deployment-lab-With-FS") as mlflow_run:

    ## Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True)

    clf.fit(X_train_pdf, Y_train_pdf)

    ## Infer output schema
    try:
        output_schema = _infer_schema(Y_train_pdf)
    except Exception as e:
        warnings.warn(f"Could not infer model output schema: {e}")
        output_schema = None

    model_name = f"{DA.catalog_name}.{DA.schema_name}.ml_model"

    ## Log using feature engineering client and push to registry
    fe.log_model(
        model=clf,
        artifact_path="decision_tree",
        flavor=mlflow.sklearn,
        training_set=training_set_spec,
        output_schema=output_schema,
        registered_model_name= model_name
    )

    ## Set model alias (i.e. Champion)
    client.set_registered_model_alias(model_name, "Champion", get_latest_model_version(model_name))
```

</div>
</details>

**Step 4: Use the Model for Inference**
  + Join the lookup DataFrame with the feature table on the primary key to retrieve the full feature set.
  + Wrap the trained model in a `pandas_udf` and apply it to the feature-enriched DataFrame for distributed batch inference.

In [0]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

## Load the model
model_uri = f"models:/{model_name}@champion"

## Define the test dataset (lookup key only)
test_features_df = test_df.select("customerID")

## Retrieve features from feature table
feature_table_df = spark.table(feature_table_name)
lookup_with_features_df = test_features_df.join(feature_table_df, on="customerID", how="left")

## Use the trained sklearn model as a UDF closure variable for distributed inference
@pandas_udf(StringType())
def predict_func(*cols):
    import pandas as pd
    input_df = pd.DataFrame({name: col for name, col in zip(features, cols)})
    return pd.Series(clf.predict(input_df).astype(str))

## Perform batch inference using feature-enriched lookup df
result_df = lookup_with_features_df.withColumn("prediction",predict_func(*[lookup_with_features_df[f] for f in features]))

## Display the inference results
display(result_df)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Step 4: Use the Model for Inference
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

## Load the model
model_uri = f"models:/{model_name}@champion"

## Define the test dataset (lookup key only)
test_features_df = test_df.select("customerID")

## Retrieve features from feature table (replicates score_batch feature lookup)
feature_table_df = spark.table(feature_table_name)
lookup_with_features_df = test_features_df.join(feature_table_df, on="customerID", how="left")

## clf is the trained sklearn model in memory — picklable and works as a UDF closure variable
@pandas_udf(StringType())
def predict_func(*cols):
    import pandas as pd
    input_df = pd.DataFrame({name: col for name, col in zip(features, cols)})
    return pd.Series(clf.predict(input_df).astype(str))

## Perform batch inference using feature-enriched lookup df
result_df = lookup_with_features_df.withColumn(
    "prediction",
    predict_func(*[lookup_with_features_df[f] for f in features])
)

## Display the inference results
display(result_df)
```

</div>
</details>

## Task 3: Assess Liquid Clustering:

Evaluate the performance results for specific optimization techniques, such as: Liquid Clustering Follow the step-wise instructions below:  
+ **Step 1:** Create `batch_inference_liquid_clustering` table and import the following columns: `customerID`, `Churn`, `SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges`, and `prediction`.
+ **Step 2:**  Begin by assessing Liquid Clustering, an optimization technique for improving performance by physically organizing data based on a specified clustering column.
+ **Step 3:**  Optimize the target table for Liquid Clustering.
+ **Step 4:** Specify the `CLUSTER BY` clause with the desired columns (e.g., (customerID, tenure)) to enable Liquid Clustering on the table.

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_catalog.dbx_schema.batch_inference_liquid_clustering(
    customerID STRING,
  Churn STRING,
  SeniorCitizen DOUBLE,
  tenure DOUBLE,
  MonthlyCharges DOUBLE,
  TotalCharges DOUBLE,
  prediction STRING
  )


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 3: Assess Liquid Clustering
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
%sql
CREATE OR REPLACE TABLE batch_inference_liquid_clustering(
  customerID STRING,
  Churn STRING,
  SeniorCitizen DOUBLE,
  tenure DOUBLE,
  MonthlyCharges DOUBLE,
  TotalCharges DOUBLE,
  prediction STRING
  )
```

</div>
</details>

In [0]:
%sql
OPTIMIZE dbx_catalog.dbx_schema.batch_inference_liquid_clustering;
ALTER TABLE dbx_catalog.dbx_schema.batch_inference_liquid_clustering
CLUSTER BY (customerID, tenure);


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 3: Assess Liquid Clustering
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
%sql
OPTIMIZE batch_inference_liquid_clustering;
ALTER TABLE batch_inference_liquid_clustering
CLUSTER BY (customerID, tenure);
```

</div>
</details>

In [0]:
result_df.write \
    .format("delta") \
    .mode("append") \
    .clusterBy("customerID", "tenure") \
    .saveAsTable("dbx_catalog.dbx_schema.batch_inference_liquid_clustering")


## Conclusion

This lab provides you with hands-on experience in batch deployment, covering model inference, Model Registry usage, and the impact of features like Liquid Clustering on performance. you will gain practical insights into deploying models at scale in a batch-oriented environment.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
